In [1]:

%pip install -q ucimlrepo xgboost imbalanced-learn


In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Download UCI Early Stage Diabetes Risk Prediction dataset
diabetes = fetch_ucirepo(id=529)

# Extract features and target
X_new = diabetes.data.features
y_new = diabetes.data.targets

print("Dataset shape:", X_new.shape)
print("Target shape:", y_new.shape)

print("\nFirst 5 rows:")
display(X_new.head())

print("\nTarget distribution:")
print(y_new.value_counts())

print("\nMissing values:")
print(X_new.isnull().sum())

Dataset shape: (520, 16)
Target shape: (520, 1)

First 5 rows:


,age,gender,polyuria,polydipsia,sudden_weight_loss,weakness,polyphagia,genital_thrush,visual_blurring,itching,irritability,delayed_healing,partial_paresis,muscle_stiffness,alopecia,obesity
0,40,Male,No,Yes,No,Yes,No,No,No,Yes,No,Yes,No,Yes,Yes,Yes
1,58,Male,No,No,No,Yes,No,No,Yes,No,No,No,Yes,No,Yes,No
2,41,Male,Yes,No,No,Yes,Yes,No,No,Yes,No,Yes,No,Yes,Yes,No
3,45,Male,No,No,Yes,Yes,Yes,Yes,No,Yes,No,Yes,No,No,No,No
4,60,Male,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes



Target distribution:
class   
Positive    320
Negative    200
Name: count, dtype: int64

Missing values:
age                   0
gender                0
polyuria              0
polydipsia            0
sudden_weight_loss    0
weakness              0
polyphagia            0
genital_thrush        0
visual_blurring       0
itching               0
irritability          0
delayed_healing       0
partial_paresis       0
muscle_stiffness      0
alopecia              0
obesity               0
dtype: int64


## Data-quality audit — inspect duplicates before model evaluation

In [3]:
import pandas as pd

# Combine features and target to inspect complete records
dataset_check = X_new.copy()
dataset_check["Target"] = y_new.iloc[:, 0].values

# Count exact duplicates
duplicate_count = dataset_check.duplicated().sum()

# Check whether identical feature profiles occur more than once
feature_duplicates = X_new.duplicated().sum()

# Check for identical features with conflicting labels
conflicting_profiles = (
    dataset_check.groupby(list(X_new.columns))["Target"]
    .nunique()
    .gt(1)
    .sum()
)

print("=== DATA QUALITY CHECK ===")
print("Total records:", len(dataset_check))
print("Exact duplicate records:", duplicate_count)
print("Repeated feature profiles:", feature_duplicates)
print("Feature profiles with conflicting labels:", conflicting_profiles)

=== DATA QUALITY CHECK ===
Total records: 520
Exact duplicate records: 269
Repeated feature profiles: 269
Feature profiles with conflicting labels: 0


## Exploratory baseline — original 520 rows (duplicates retained)

**Caution:** identical records can appear in both training and testing sets, inflating apparent generalisation. Keep this baseline only to illustrate the duplicate problem.

In [4]:
# If needed in Colab, uncomment the next line:
# %pip -q install xgboost imbalanced-learn

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

# Prepare the target labels
y = y_new.iloc[:, 0].map({
    "Negative": 0,
    "Positive": 1
})

# Identify categorical and numerical features
categorical_cols = X_new.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_cols = X_new.select_dtypes(
    include=["number"]
).columns.tolist()

# Split before preprocessing to avoid data leakage
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_new,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Encode categorical features using training data only
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              sparse_output=False), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

# ADASYN is applied only within the training pipeline
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("adasyn", ADASYN(random_state=42)),
    ("classifier", XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    ))
])

# Train the model
model.fit(X_train_new, y_train_new)

# Evaluate on untouched test data
y_pred = model.predict(X_test_new)
y_prob = model.predict_proba(X_test_new)[:, 1]

print("=== NEW DATASET: XGBOOST + ADASYN ===")
print("Training records:", len(X_train_new))
print("Testing records:", len(X_test_new))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_new, y_pred))

print("\nAccuracy:", round(accuracy_score(y_test_new, y_pred), 4))
print("\nROC-AUC:", round(roc_auc_score(y_test_new, y_prob), 4))

print("\nClassification Report:")
print(classification_report(y_test_new, y_pred))

=== NEW DATASET: XGBOOST + ADASYN ===
Training records: 416
Testing records: 104

Confusion Matrix:
[[40  0]
 [ 2 62]]

Accuracy: 0.9808

ROC-AUC: 0.9992

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98        40
           1       1.00      0.97      0.98        64

    accuracy                           0.98       104
   macro avg       0.98      0.98      0.98       104
weighted avg       0.98      0.98      0.98       104



## Exploratory five-fold CV — duplicate-affected

These folds are formed from the uncleaned data. Use the duplicate-aware CV below for the primary report results.

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_new,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    n_jobs=-1
)

print("=== 5-FOLD CROSS-VALIDATION RESULTS ===")

for metric in ["accuracy", "f1", "roc_auc"]:
    values = scores[f"test_{metric}"]
    print(f"{metric.upper()}:")
    print("Individual folds:", values.round(4))
    print(f"Mean: {values.mean():.4f}")
    print(f"Standard deviation: {values.std():.4f}")
    print()

=== 5-FOLD CROSS-VALIDATION RESULTS ===
ACCURACY:
Individual folds: [0.9615 0.9808 0.9712 0.9808 0.9615]
Mean: 0.9712
Standard deviation: 0.0086

F1:
Individual folds: [0.9683 0.9846 0.976  0.9841 0.9683]
Mean: 0.9763
Standard deviation: 0.0072

ROC_AUC:
Individual folds: [0.9887 0.9965 0.9977 0.9961 0.9938]
Mean: 0.9945
Standard deviation: 0.0032



## Primary evaluation — remove exact duplicates before splitting

This section reports a stratified 80/20 held-out test and five-fold cross-validation after deduplication. ADASYN is applied only to each training partition.

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

# Remove exact duplicate records before splitting
clean_data = dataset_check.drop_duplicates().reset_index(drop=True)

X_clean = clean_data.drop(columns=["Target"])
y_clean = clean_data["Target"].astype(str)

print("Original records:", len(dataset_check))
print("Unique records:", len(clean_data))
print("Duplicates removed:", len(dataset_check) - len(clean_data))
print("\nClass distribution:")
print(y_clean.value_counts())

# Identify numerical and categorical columns
numeric_cols = X_clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_clean.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore",
                                  sparse_output=False))
    ]), categorical_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("adasyn", ADASYN(random_state=42, n_neighbors=3)),
    ("xgboost", XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    ))
])

# Stratified train/test split after deduplication
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean, y_clean,
    test_size=0.20,
    random_state=42,
    stratify=y_clean
)

# XGBoost requires numeric target labels
label_mapping = {"Negative": 0, "Positive": 1}

y_train_num = y_train_clean.map(label_mapping)
y_test_num = y_test_clean.map(label_mapping)

model.fit(X_train_clean, y_train_num)

pred = model.predict(X_test_clean)
prob = model.predict_proba(X_test_clean)[:, 1]

print("\n=== DUPLICATE-AWARE TEST RESULTS ===")
print("Training records:", len(X_train_clean))
print("Testing records:", len(X_test_clean))
print("Accuracy:", round(accuracy_score(y_test_num, pred), 4))
print("F1-score:", round(f1_score(y_test_num, pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test_num, prob), 4))
print("Confusion matrix:")
print(confusion_matrix(y_test_num, pred))

# Five-fold cross-validation with preprocessing
# and ADASYN inside each training fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_validate(
    model,
    X_clean,
    y_clean.map(label_mapping),
    cv=cv,
    scoring=["accuracy", "f1", "roc_auc"],
    n_jobs=1
)

print("\n=== DUPLICATE-AWARE 5-FOLD CV ===")
for metric in ["accuracy", "f1", "roc_auc"]:
    values = cv_scores["test_" + metric]
    print(
        metric,
        "Mean:", round(values.mean(), 4),
        "SD:", round(values.std(), 4)
    )

Original records: 520
Unique records: 251
Duplicates removed: 269

Class distribution:
Target
Positive    173
Negative     78
Name: count, dtype: int64

=== DUPLICATE-AWARE TEST RESULTS ===
Training records: 200
Testing records: 51
Accuracy: 0.8824
F1-score: 0.9062
ROC-AUC: 0.9661
Confusion matrix:
[[16  0]
 [ 6 29]]

=== DUPLICATE-AWARE 5-FOLD CV ===
accuracy Mean: 0.9204 SD: 0.012
f1 Mean: 0.9423 SD: 0.0082
roc_auc Mean: 0.9663 SD: 0.0273


## Reporting notes

- The original notebook recorded **98.08% accuracy** on duplicate-affected data, versus **88.24% accuracy**, **0.9062 F1**, and **0.9661 ROC-AUC** after deduplication. These are *previously recorded* values; rerun this cleaned notebook to obtain current outputs.
- There were **269 exact duplicate rows**, leaving **251 unique rows** in the prior run. Report the primary deduplicated result, the test-set size, and the class distribution.
- This is a held-out observational dataset, not a clinical validation. Duplicates, dataset size, and differences from the original paper constrain conclusions.